In [1]:

from pathlib import Path
import duckdb
import pandas as pd

# Locate the repository root, whether the notebook runs from
# the notebooks/ directory or the repository root.
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

# Canonical data lives here; A02 will eventually be written
# to data/analytical/, separately from the canonical sources.
canonical_dir = project_root / "data" / "canonical"

con = duckdb.connect()

# Confirm the files and directories before proceeding.
print("Project root:", project_root)
print("Canonical directory:", canonical_dir)
print("Canonical directory exists:", canonical_dir.is_dir())

for filename in ["player_team_seasons.csv", "player_salaries.csv"]:
    path = canonical_dir / filename
    print(f"{filename}: {'FOUND' if path.is_file() else 'MISSING'}")


Project root: /Users/jonathantran/Desktop/nba_economics
Canonical directory: /Users/jonathantran/Desktop/nba_economics/data/canonical
Canonical directory exists: True
player_team_seasons.csv: FOUND
player_salaries.csv: FOUND


In [3]:

sources = {
    "C05": canonical_dir / "player_team_seasons.csv",
    "C07": canonical_dir / "player_salaries.csv",
}

for name, path in sources.items():

    # Convert the Path object into a SQL-safe string literal.
    # This is appropriate here because these paths are defined
    # locally in our code, not supplied by external users.
    sql_path = str(path).replace("'", "''")

    # Create a temporary DuckDB view for each canonical CSV.
    # We embed the escaped file path directly because CREATE VIEW
    # cannot use the ? parameter placeholder in this context.
    con.execute(
        f"""
        CREATE OR REPLACE TEMP VIEW {name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
        """
    )

    print(f"\n{'=' * 50}")
    print(f"{name} — {path.name}")

    # Inspect the column names and inferred data types.
    print("\nSCHEMA:")
    print(con.execute(f"DESCRIBE {name}").df().to_string(index=False))

    # Count the number of records.
    print("\nROW COUNT:")
    print(con.execute(f"SELECT COUNT(*) AS n_rows FROM {name}").df())

    # Preview the first three observations.
    print("\nSAMPLE:")
    print(con.execute(f"SELECT * FROM {name} LIMIT 3").df().to_string(index=False))



C05 — player_team_seasons.csv

SCHEMA:
                      column_name column_type null  key default extra
                        player_id      BIGINT  YES None    None  None
                     franchise_id      BIGINT  YES None    None  None
                           season     VARCHAR  YES None    None  None
                              age      DOUBLE  YES None    None  None
                    height_inches      DOUBLE  YES None    None  None
                       weight_lbs     VARCHAR  YES None    None  None
                     games_played      BIGINT  YES None    None  None
                     points_total      BIGINT  YES None    None  None
                   rebounds_total      BIGINT  YES None    None  None
                    assists_total      BIGINT  YES None    None  None
                             wins      BIGINT  YES None    None  None
                           losses      BIGINT  YES None    None  None
                          win_pct      DOUBLE  YES

In [4]:

# ------------------------------------------------------------
# A02: Structural validation of C05 and C07
# ------------------------------------------------------------

# Check whether either canonical source contains duplicate
# player-franchise-season keys.
for source in ["C05", "C07"]:

    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT (player_id, franchise_id, season))
                AS distinct_keys,
            COUNT(*) - COUNT(DISTINCT (player_id, franchise_id, season))
                AS duplicate_key_rows,
            COUNT(*) FILTER (
                WHERE player_id IS NULL
                   OR franchise_id IS NULL
                   OR season IS NULL
            ) AS null_key_rows
        FROM {source}
    """).df()

    print(f"\n{source} — KEY VALIDATION")
    print(result.to_string(index=False))


# Check whether the same player is associated with more than
# one recorded franchise during a single season.
multi_franchise = con.execute("""
    SELECT
        player_id,
        season,
        COUNT(DISTINCT franchise_id) AS franchise_count
    FROM C05
    GROUP BY player_id, season
    HAVING COUNT(DISTINCT franchise_id) > 1
    ORDER BY franchise_count DESC, season, player_id
""").df()

print("\nC05 — MULTIPLE FRANCHISES PER PLAYER-SEASON")
print("Affected player-seasons:", len(multi_franchise))
print(multi_franchise.head(10).to_string(index=False))


# Check C05-to-C07 join cardinality and salary coverage.
# LEFT JOIN preserves all C05 observations unless C07
# contains multiple matching salary records.
join_audit = con.execute("""
    SELECT
        COUNT(*) AS joined_rows,
        COUNT(*) FILTER (
            WHERE s.player_id IS NOT NULL
        ) AS matched_rows,
        COUNT(*) FILTER (
            WHERE s.player_id IS NULL
        ) AS unmatched_rows,
        COUNT(*) FILTER (
            WHERE s.salary IS NOT NULL
        ) AS salary_nonnull_rows
    FROM C05 AS p
    LEFT JOIN C07 AS s
        ON p.player_id = s.player_id
       AND p.franchise_id = s.franchise_id
       AND p.season = s.season
""").df()

print("\nC05 → C07 — JOIN AUDIT")
print(join_audit.to_string(index=False))



C05 — KEY VALIDATION
 total_rows  distinct_keys  duplicate_key_rows  null_key_rows
      12810          12810                   0              0

C07 — KEY VALIDATION
 total_rows  distinct_keys  duplicate_key_rows  null_key_rows
      11826          11826                   0              0

C05 — MULTIPLE FRANCHISES PER PLAYER-SEASON
Affected player-seasons: 0
Empty DataFrame
Columns: [player_id, season, franchise_count]
Index: []

C05 → C07 — JOIN AUDIT
 joined_rows  matched_rows  unmatched_rows  salary_nonnull_rows
       12810         10601            2209                10601


In [5]:

# ------------------------------------------------------------
# A02: Inspect essential player and season dimensions
# ------------------------------------------------------------

additional_sources = {
    "C03": canonical_dir / "players.csv",
    "C01": canonical_dir / "seasons.csv",
}

for name, path in additional_sources.items():

    # Ensure the canonical file exists before reading it.
    if not path.is_file():
        raise FileNotFoundError(f"Missing canonical source: {path}")

    # Escape single quotes so the local file path can be
    # safely embedded in the DuckDB SQL statement.
    sql_path = str(path).replace("'", "''")

    # Create a temporary view; canonical CSVs remain unchanged.
    con.execute(f"""
        CREATE OR REPLACE TEMP VIEW {name} AS
        SELECT *
        FROM read_csv_auto('{sql_path}')
    """)

    print(f"\n{'=' * 55}")
    print(f"{name} — {path.name}")

    # Display all available columns and inferred types.
    print("\nSCHEMA:")
    print(con.execute(f"DESCRIBE {name}").df().to_string(index=False))

    # Confirm record counts.
    print("\nROW COUNT:")
    print(
        con.execute(
            f"SELECT COUNT(*) AS n_rows FROM {name}"
        ).df().to_string(index=False)
    )

    # Preview representative source records.
    print("\nSAMPLE:")
    print(
        con.execute(
            f"SELECT * FROM {name} LIMIT 3"
        ).df().to_string(index=False)
    )



C03 — players.csv

SCHEMA:
 column_name column_type null  key default extra
   player_id      BIGINT  YES None    None  None
 player_name     VARCHAR  YES None    None  None
     country     VARCHAR  YES None    None  None
     college     VARCHAR  YES None    None  None
  draft_year     VARCHAR  YES None    None  None
 draft_round     VARCHAR  YES None    None  None
draft_number     VARCHAR  YES None    None  None

ROW COUNT:
 n_rows
   2584

SAMPLE:
 player_id     player_name country          college draft_year draft_round draft_number
         3      Grant Long     USA Eastern Michigan       1988           2           33
        15 Eric Piatkowski     USA         Nebraska       1994           1           15
        21    Greg Anthony     USA             UNLV       1991           1           12

C01 — seasons.csv

SCHEMA:
         column_name column_type null  key default extra
              season     VARCHAR  YES None    None  None
                Year      BIGINT  YES None    Non

In [6]:

# ------------------------------------------------------------
# A02: Validate player and season dimensions
# ------------------------------------------------------------

# Each dimension should have one record per join key.
for source, key in [
    ("C03", "player_id"),
    ("C01", "season")
]:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT {key}) AS unique_keys,
            COUNT(*) - COUNT(DISTINCT {key}) AS duplicate_keys,
            COUNT(*) FILTER (WHERE {key} IS NULL) AS null_keys
        FROM {source}
    """).df()

    print(f"\n{source} — DIMENSION KEY VALIDATION")
    print(result.to_string(index=False))


# Check whether every C05 player has a corresponding
# player record in C03, without discarding unmatched rows.
player_coverage = con.execute("""
    SELECT
        COUNT(*) AS c05_rows,
        COUNT(*) FILTER (
            WHERE d.player_id IS NOT NULL
        ) AS matched_players,
        COUNT(*) FILTER (
            WHERE d.player_id IS NULL
        ) AS unmatched_players
    FROM C05 AS p
    LEFT JOIN C03 AS d
        ON p.player_id = d.player_id
""").df()

print("\nC05 → C03 — PLAYER JOIN")
print(player_coverage.to_string(index=False))


# Check whether every C05 season maps to C01.
season_coverage = con.execute("""
    SELECT
        COUNT(*) AS c05_rows,
        COUNT(*) FILTER (
            WHERE s.season IS NOT NULL
        ) AS matched_seasons,
        COUNT(*) FILTER (
            WHERE s.season IS NULL
        ) AS unmatched_seasons
    FROM C05 AS p
    LEFT JOIN C01 AS s
        ON p.season = s.season
""").df()

print("\nC05 → C01 — SEASON JOIN")
print(season_coverage.to_string(index=False))



C03 — DIMENSION KEY VALIDATION
 total_rows  unique_keys  duplicate_keys  null_keys
       2584         2584               0          0

C01 — DIMENSION KEY VALIDATION
 total_rows  unique_keys  duplicate_keys  null_keys
         26           26               0          0

C05 → C03 — PLAYER JOIN
 c05_rows  matched_players  unmatched_players
    12810            12810                  0

C05 → C01 — SEASON JOIN
 c05_rows  matched_seasons  unmatched_seasons
    12810            12810                  0


In [7]:

# ------------------------------------------------------------
# A02 v0: Player–franchise–season analytical base
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v0_base AS

    SELECT
        -- Preserve the complete player-performance record.
        p.*,

        -- C03: Player identity and draft background.
        d.player_name,
        d.country,
        d.college,
        d.draft_year,
        d.draft_round,
        d.draft_number,

        -- C07: Salary and explicit availability indicator.
        s.salary,
        CASE
            WHEN s.salary IS NOT NULL THEN 1
            ELSE 0
        END AS salary_available,

        -- C01: Financial context for the relevant season.
        c."Salary Cap" AS salary_cap,
        c."Luxury Tax" AS luxury_tax_threshold,
        c."1st Apron" AS first_apron_threshold,
        c."2nd Apron" AS second_apron_threshold,
        c.cba_id

    FROM C05 AS p

    LEFT JOIN C03 AS d
        ON p.player_id = d.player_id

    LEFT JOIN C07 AS s
        ON p.player_id = s.player_id
       AND p.franchise_id = s.franchise_id
       AND p.season = s.season

    LEFT JOIN C01 AS c
        ON p.season = c.season
""")

print("A02 v0 created successfully.")

# Check the size of the new analytical view.
print("\nA02 v0 DIMENSIONS:")
print(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(*) FILTER (WHERE salary_available = 1)
            AS salary_matched,
        COUNT(*) FILTER (WHERE salary_available = 0)
            AS salary_unmatched
    FROM a02_v0_base
""").df().to_string(index=False))

# Inspect a small selection of columns to verify the joins.
print("\nA02 v0 SAMPLE:")
print(con.execute("""
    SELECT
        player_name,
        season,
        franchise_id,
        games_played,
        points_per_game,
        salary,
        salary_cap,
        salary_available
    FROM a02_v0_base
    ORDER BY season, player_name
    LIMIT 10
""").df().to_string(index=False))


A02 v0 created successfully.

A02 v0 DIMENSIONS:
 rows  unique_keys  salary_matched  salary_unmatched
12810        12810           10601              2209

A02 v0 SAMPLE:
   player_name    season  franchise_id  games_played  points_per_game  salary  salary_cap  salary_available
    A.C. Green 2000-2001    1610612748            82              4.5 2250000    35500000                 1
   A.J. Guyton 2000-2001    1610612741            33              6.0  316000    35500000                 1
   Aaron McKie 2000-2001    1610612755            76             11.6 1820000    35500000                 1
Aaron Williams 2000-2001    1610612751            82             10.2 2250000    35500000                 1
    Adam Keefe 2000-2001    1610612744            67              2.5 3390000    35500000                 1
  Adonal Foyle 2000-2001    1610612744            58              5.9 3578000    35500000                 1
Adrian Griffin 2000-2001    1610612738            44              2.1  49

In [8]:

# ------------------------------------------------------------
# A02 v1 — Pass A: Salary normalization and comparisons
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation_a AS

    WITH salary_benchmarks AS (
        SELECT
            season,
            franchise_id,

            -- League benchmarks repeat across franchise groups,
            -- so calculate league-wide values with window functions.
            AVG(salary) OVER (
                PARTITION BY season
            ) AS league_mean_salary,

            MEDIAN(salary) OVER (
                PARTITION BY season
            ) AS league_median_salary,

            -- Franchise-level benchmarks.
            AVG(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS team_mean_salary,

            MEDIAN(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS team_median_salary,

            player_id,
            salary

        FROM a02_v0_base
    ),

    salary_features AS (
        SELECT
            p.*,

            -- Historical normalization against the salary cap.
            p.salary * 1.0 / NULLIF(p.salary_cap, 0)
                AS salary_to_cap_ratio,

            -- Rank within the recorded franchise-season.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    RANK() OVER (
                        PARTITION BY p.franchise_id, p.season
                        ORDER BY p.salary DESC NULLS LAST
                    )
            END AS salary_rank_franchise_season,

            -- Rank across the entire league in the same season.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    RANK() OVER (
                        PARTITION BY p.season
                        ORDER BY p.salary DESC NULLS LAST
                    )
            END AS salary_rank_league_season,

            -- Salary percentile among matched league salaries.
            CASE
                WHEN p.salary IS NOT NULL THEN
                    (
                        RANK() OVER (
                            PARTITION BY p.season
                            ORDER BY p.salary ASC NULLS LAST
                        ) - 1
                    ) * 1.0
                    / NULLIF(
                        COUNT(p.salary) OVER (
                            PARTITION BY p.season
                        ) - 1,
                        0
                    )
            END AS salary_percentile_league_season

        FROM a02_v0_base AS p
    )

    SELECT
        f.*,

        -- Reference statistics retained for interpretability.
        b.league_mean_salary,
        b.league_median_salary,
        b.team_mean_salary,
        b.team_median_salary,

        -- Player salary relative to each benchmark.
        f.salary * 1.0 / NULLIF(b.league_mean_salary, 0)
            AS salary_vs_league_mean,

        f.salary * 1.0 / NULLIF(b.league_median_salary, 0)
            AS salary_vs_league_median,

        f.salary * 1.0 / NULLIF(b.team_mean_salary, 0)
            AS salary_vs_team_mean,

        f.salary * 1.0 / NULLIF(b.team_median_salary, 0)
            AS salary_vs_team_median

    FROM salary_features AS f

    LEFT JOIN salary_benchmarks AS b
        ON f.player_id = b.player_id
       AND f.franchise_id = b.franchise_id
       AND f.season = b.season
""")

print("A02 v1 Pass A created successfully.")

# ------------------------------------------------------------
# Validate that no player-season records were duplicated.
# ------------------------------------------------------------

result = con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(salary_to_cap_ratio) AS cap_ratios,
        COUNT(salary_rank_league_season) AS league_ranks,
        COUNT(salary_vs_league_median) AS league_median_ratios,
        COUNT(salary_vs_team_median) AS team_median_ratios
    FROM a02_v1_compensation_a
""").df()

print("\nA02 v1 PASS A VALIDATION:")
print(result.to_string(index=False))


A02 v1 Pass A created successfully.

A02 v1 PASS A VALIDATION:
 rows  unique_keys  cap_ratios  league_ranks  league_median_ratios  team_median_ratios
12810        12810       10601         10601                 10601               10601


In [9]:
# ------------------------------------------------------------
# A02 v1 Pass A — Semantic validation
# ------------------------------------------------------------

# Inspect the league's highest-paid players in a recent season.
top_salaries = con.execute("""
    SELECT
        player_name,
        season,
        salary,
        salary_rank_league_season,
        salary_rank_franchise_season,
        ROUND(salary_to_cap_ratio, 3) AS salary_to_cap,
        ROUND(salary_percentile_league_season, 3) AS league_percentile,
        ROUND(salary_vs_league_median, 2) AS vs_league_median,
        ROUND(salary_vs_team_median, 2) AS vs_team_median
    FROM a02_v1_compensation_a
    WHERE season = '2024-2025'
      AND salary IS NOT NULL
    ORDER BY salary DESC
    LIMIT 10
""").df()

print("TOP SALARIES — 2024-2025")
print(top_salaries.to_string(index=False))


# Check for invalid ratios, rankings, and percentiles.
validation = con.execute("""
    SELECT
        COUNT(*) FILTER (
            WHERE salary_to_cap_ratio < 0
        ) AS negative_cap_ratios,

        COUNT(*) FILTER (
            WHERE salary_percentile_league_season < 0
               OR salary_percentile_league_season > 1
        ) AS invalid_percentiles,

        COUNT(*) FILTER (
            WHERE salary IS NOT NULL
              AND salary_rank_league_season < 1
        ) AS invalid_league_ranks,

        COUNT(*) FILTER (
            WHERE salary IS NOT NULL
              AND salary_rank_franchise_season < 1
        ) AS invalid_team_ranks,

        COUNT(*) FILTER (
            WHERE salary IS NULL
              AND (
                  salary_rank_league_season IS NOT NULL
                  OR salary_percentile_league_season IS NOT NULL
              )
        ) AS unmatched_salary_with_rank

    FROM a02_v1_compensation_a
""").df()

print("\nFEATURE VALIDATION")
print(validation.to_string(index=False))

TOP SALARIES — 2024-2025
       player_name    season   salary  salary_rank_league_season  salary_rank_franchise_season  salary_to_cap  league_percentile  vs_league_median  vs_team_median
     Stephen Curry 2024-2025 55761216                          1                             1          0.397              1.000             10.79            8.30
       Joel Embiid 2024-2025 51415938                          2                             1          0.366              0.996              9.95           18.69
      Nikola Jokić 2024-2025 51415938                          2                             1          0.366              0.996              9.95           16.08
      Kevin Durant 2024-2025 51179021                          4                             1          0.364              0.993              9.90           10.24
      Bradley Beal 2024-2025 50203930                          5                             2          0.357              0.991              9.71           10.

In [10]:
# ------------------------------------------------------------
# A02 v1 — Pass B: Salary allocation and distribution
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation_b AS

    WITH salary_pools AS (
        SELECT
            *,

            -- Total successfully matched salaries for each
            -- recorded franchise-season.
            SUM(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS matched_roster_salary_total,

            -- Total matched salaries across the entire league
            -- for the same season.
            SUM(salary) OVER (
                PARTITION BY season
            ) AS matched_league_salary_total,

            -- Count of salary-matched players on each
            -- recorded franchise-season.
            COUNT(salary) OVER (
                PARTITION BY franchise_id, season
            ) AS matched_roster_salary_count,

            -- Total player records for the franchise-season,
            -- including those without matched salaries.
            COUNT(*) OVER (
                PARTITION BY franchise_id, season
            ) AS recorded_roster_player_count

        FROM a02_v1_compensation_a
    )

    SELECT
        *,

        -- Share of the franchise's matched salary pool.
        salary * 1.0 /
            NULLIF(matched_roster_salary_total, 0)
            AS salary_share_recorded_roster,

        -- Share of all matched league salaries that season.
        salary * 1.0 /
            NULLIF(matched_league_salary_total, 0)
            AS salary_share_league,

        -- Fraction of recorded franchise-season players
        -- with successfully matched salary records.
        matched_roster_salary_count * 1.0 /
            NULLIF(recorded_roster_player_count, 0)
            AS roster_salary_match_rate

    FROM salary_pools
""")

print("A02 v1 Pass B created successfully.")

print("\nA02 v1 PASS B VALIDATION:")

print(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT (player_id, franchise_id, season))
            AS unique_keys,
        COUNT(salary_share_recorded_roster)
            AS roster_shares,
        COUNT(salary_share_league)
            AS league_shares,
        MIN(roster_salary_match_rate)
            AS min_roster_match_rate,
        MAX(roster_salary_match_rate)
            AS max_roster_match_rate
    FROM a02_v1_compensation_b
""").df().to_string(index=False))

A02 v1 Pass B created successfully.

A02 v1 PASS B VALIDATION:
 rows  unique_keys  roster_shares  league_shares  min_roster_match_rate  max_roster_match_rate
12810        12810          10601          10601               0.066667                    1.0


In [11]:
# ------------------------------------------------------------
# A02 v1 — Identify low-coverage franchise-seasons
# ------------------------------------------------------------

low_coverage = con.execute("""
    SELECT
        season,
        franchise_id,
        MAX(recorded_roster_player_count) AS recorded_players,
        MAX(matched_roster_salary_count) AS matched_players,
        ROUND(MAX(roster_salary_match_rate), 3) AS match_rate,
        MAX(matched_roster_salary_total) AS matched_salary_total
    FROM a02_v1_compensation_b
    GROUP BY season, franchise_id
    ORDER BY match_rate ASC, season
    LIMIT 15
""").df()

print("LOWEST SALARY-MATCH COVERAGE:")
print(low_coverage.to_string(index=False))

LOWEST SALARY-MATCH COVERAGE:
   season  franchise_id  recorded_players  matched_players  match_rate  matched_salary_total
2004-2005    1610612765                15                1       0.067             6000000.0
2004-2005    1610612738                12                1       0.083            14625000.0
2004-2005    1610612746                17                2       0.118            12086000.0
2004-2005    1610612764                16                2       0.125             8404000.0
2004-2005    1610612762                16                2       0.125             8047000.0
2004-2005    1610612741                15                2       0.133            15150000.0
2004-2005    1610612754                20                3       0.150            15606000.0
2004-2005    1610612740                18                3       0.167             7864000.0
2003-2004    1610612762                17                3       0.176            21290000.0
2004-2005    1610612766                1

In [12]:
# ------------------------------------------------------------
# A02 v1 — Salary coverage by season
# ------------------------------------------------------------

season_coverage = con.execute("""
    SELECT
        season,

        -- Total player-franchise-season observations.
        COUNT(*) AS recorded_players,

        -- Observations with successfully matched salaries.
        COUNT(salary) AS matched_players,

        -- Share of observations with matched salaries.
        ROUND(
            COUNT(salary) * 1.0 / NULLIF(COUNT(*), 0),
            3
        ) AS salary_match_rate,

        -- Total observed salaries for this season.
        ROUND(SUM(salary) / 1000000.0, 2)
            AS matched_salary_total_millions

    FROM a02_v1_compensation_b

    GROUP BY season
    ORDER BY season
""").df()

print("SALARY COVERAGE BY SEASON:")
print(season_coverage.to_string(index=False))

SALARY COVERAGE BY SEASON:
   season  recorded_players  matched_players  salary_match_rate  matched_salary_total_millions
2000-2001               441              369              0.837                        1293.90
2001-2002               440              411              0.934                        1398.28
2002-2003               428              278              0.650                        1321.13
2003-2004               442              177              0.400                        1040.09
2004-2005               464              118              0.254                         813.44
2005-2006               458              420              0.917                        1696.24
2006-2007               458              432              0.943                        1728.04
2007-2008               451              429              0.951                        1823.52
2008-2009               445              427              0.960                        2000.96
2009-2010              

In [13]:
# ------------------------------------------------------------
# A02 v1 — Final compensation view and coverage validation
# ------------------------------------------------------------

con.execute("""
    CREATE OR REPLACE TEMP VIEW a02_v1_compensation AS

    SELECT
        *,

        -- Number of salary-matched player observations
        -- across the league in the same season.
        COUNT(salary) OVER (
            PARTITION BY season
        ) AS matched_league_salary_count,

        -- Proportion of league player observations
        -- with matched salary records.
        COUNT(salary) OVER (
            PARTITION BY season
        ) * 1.0
        / NULLIF(
            COUNT(*) OVER (PARTITION BY season),
            0
        ) AS league_salary_match_rate

    FROM a02_v1_compensation_b
""")

# Each franchise-season's observed salary shares should
# sum to 1, provided at least one salary is matched.
share_validation = con.execute("""
    WITH team_shares AS (
        SELECT
            season,
            franchise_id,
            SUM(salary_share_recorded_roster)
                AS total_share
        FROM a02_v1_compensation
        GROUP BY season, franchise_id
    ),

    league_shares AS (
        SELECT
            season,
            SUM(salary_share_league)
                AS total_share
        FROM a02_v1_compensation
        GROUP BY season
    )

    SELECT
        (SELECT COUNT(*) FROM team_shares
         WHERE ABS(total_share - 1.0) > 0.000001)
            AS invalid_team_share_totals,

        (SELECT COUNT(*) FROM team_shares
         WHERE total_share IS NULL)
            AS teams_without_matched_salaries,

        (SELECT COUNT(*) FROM league_shares
         WHERE ABS(total_share - 1.0) > 0.000001)
            AS invalid_league_share_totals

""").df()

print("A02 v1 FINAL SHARE VALIDATION:")
print(share_validation.to_string(index=False))

A02 v1 FINAL SHARE VALIDATION:
 invalid_team_share_totals  teams_without_matched_salaries  invalid_league_share_totals
                         0                               0                            0


In [15]:
from pathlib import Path

# ------------------------------------------------------------
# Step 1 — Locate the existing A01 analytical Parquet file
# ------------------------------------------------------------

# Path is relative to your notebook's working directory.
analytical_dir = Path("../data/analytical")

# Find Parquet files whose names begin with "a01".
a01_files = sorted(analytical_dir.glob("a01*.parquet"))

print("A01 PARQUET FILES FOUND:")

for file in a01_files:
    print(file)

# ------------------------------------------------------------
# Step 2 — Inspect payroll-related columns
# ------------------------------------------------------------

if len(a01_files) == 1:

    # DuckDB can query Parquet directly from its file path.
    # No permanent table or temporary view is required.
    a01_path = str(a01_files[0])

    a01_columns = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [a01_path]
    ).df()

    payroll_candidates = a01_columns[
        a01_columns["column_name"].str.contains(
            "payroll|salary|taxable|cap_space|dead_money",
            case=False,
            regex=True
        )
    ]

    print("\nPOTENTIAL PAYROLL DENOMINATORS:")
    print(
        payroll_candidates[
            ["column_name", "column_type"]
        ].to_string(index=False)
    )

elif len(a01_files) == 0:
    print("No matching Parquet file found. Check the directory path.")

else:
    print("Multiple A01 files found. Select the intended analytical table.")

A01 PARQUET FILES FOUND:
../data/analytical/a01_franchise_season.parquet

POTENTIAL PAYROLL DENOMINATORS:
                                  column_name column_type
                                 team_payroll      BIGINT
                               salary_cap_hit      BIGINT
                             salary_cap_space      DOUBLE
                              taxable_payroll      DOUBLE
                                active_salary      DOUBLE
                                  dead_salary      DOUBLE
                                   salary_cap      BIGINT
                         salary_cap_floor_pct      DOUBLE
                             salary_cap_floor      BIGINT
                         payroll_to_cap_ratio      DOUBLE
                 taxable_payroll_to_tax_ratio      DOUBLE
                    tax_paid_to_payroll_ratio      DOUBLE
              estimated_cost_to_taxable_ratio      DOUBLE
                              payroll_per_win      DOUBLE
                      ta

In [17]:
# ------------------------------------------------------------
# A02 v1 — Register A01 as a DuckDB reference view
# ------------------------------------------------------------

# a01_path was established in Cell 13.
# We use a Python f-string to insert the verified file path
# directly into the SQL definition.
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW a01_reference AS
    SELECT *
    FROM read_parquet('{a01_path}')
""")

print("A01 reference view created successfully.")

# Compare one observation per franchise-season.
# MAX() is appropriate because the roster-level metrics
# repeat across every player in that franchise-season.
comparison = con.execute("""
    WITH a02_roster_totals AS (
        SELECT
            franchise_id,
            season,
            MAX(matched_roster_salary_total)
                AS matched_salary_total,
            MAX(roster_salary_match_rate)
                AS salary_match_rate,
            MAX(recorded_roster_player_count)
                AS recorded_players

        FROM a02_v1_compensation
        GROUP BY franchise_id, season
    )

    SELECT
        p.season,
        p.franchise_id,
        p.recorded_players,
        ROUND(p.salary_match_rate, 3)
            AS salary_match_rate,

        ROUND(p.matched_salary_total / 1000000.0, 2)
            AS matched_salary_millions,

        ROUND(a.active_salary / 1000000.0, 2)
            AS active_salary_millions,

        ROUND(
            p.matched_salary_total * 1.0
            / NULLIF(a.active_salary, 0),
            3
        ) AS matched_to_active_ratio

    FROM a02_roster_totals AS p

    LEFT JOIN a01_reference AS a
        ON p.franchise_id = a.franchise_id
       AND p.season = a.season

    ORDER BY p.season DESC, p.franchise_id
""").df()

print("A02 vs A01 ACTIVE SALARY COMPARISON:")
print(comparison.head(20).to_string(index=False))

print("\nMATCHED-TO-ACTIVE RATIO SUMMARY:")
print(
    comparison["matched_to_active_ratio"]
    .describe()
    .round(3)
    .to_string()
)

print("\nMISSING A01 ACTIVE SALARY VALUES:")
print(comparison["active_salary_millions"].isna().sum())

A01 reference view created successfully.
A02 vs A01 ACTIVE SALARY COMPARISON:
   season  franchise_id  recorded_players  salary_match_rate  matched_salary_millions  active_salary_millions  matched_to_active_ratio
2025-2026    1610612737                21              0.762                   181.73                  180.37                    1.008
2025-2026    1610612738                17              0.765                   189.35                  187.42                    1.010
2025-2026    1610612739                22              0.727                   221.98                  212.07                    1.047
2025-2026    1610612740                18              0.833                   182.15                  182.26                    0.999
2025-2026    1610612741                20              0.750                   176.22                  166.17                    1.060
2025-2026    1610612742                19              0.789                   139.01                  176.20   

In [19]:
from pathlib import Path
import json

# Start from the repository root.
repo_root = Path("..")

# Search terms relevant to the S08 financial-data lineage.
keywords = ["active_salary", "s08", "hoopshype"]

# Track files that cannot be read as valid notebook JSON.
skipped_files = []

for notebook_path in repo_root.rglob("*.ipynb"):

    try:
        # Open the notebook and parse its JSON structure.
        with open(notebook_path, encoding="utf-8") as f:
            notebook = json.load(f)

    except (json.JSONDecodeError, UnicodeDecodeError, OSError) as e:
        # Record the problematic file and continue searching.
        skipped_files.append((str(notebook_path), str(e)))
        continue

    # Examine code cells only, excluding notebook outputs.
    for cell_number, cell in enumerate(notebook.get("cells", []), 1):

        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))
        lines = source.splitlines()

        # Find lines containing any of our search terms.
        matches = [
            i for i, line in enumerate(lines)
            if any(word in line.lower() for word in keywords)
        ]

        if not matches:
            continue

        print(f"\nNOTEBOOK: {notebook_path}")
        print(f"CELL: {cell_number}")

        # Print three lines before and after each matching line.
        shown = set()

        for i in matches:
            for j in range(max(0, i - 3), min(len(lines), i + 4)):
                if j not in shown:
                    print(f"{j + 1:>3}: {lines[j]}")
                    shown.add(j)

# Report any notebooks that couldn't be inspected.
print("\n--- SKIPPED NOTEBOOKS ---")

if skipped_files:
    for path, error in skipped_files:
        print(f"{path}: {error}")
else:
    print("None")


NOTEBOOK: ../notebooks/c04_team_seasons.ipynb
CELL: 1
  9:     STAGING_DIR / "standings" / "s01_standings.csv"
 10: )
 11: 
 12: s08 = pd.read_csv(
 13:     STAGING_DIR / "team_payroll" / "s08_team_payroll.csv"
 14: )
 15: 
 16: s02 = pd.read_csv(STAGING_DIR / "advanced_stats" / "s02_advanced_stats.csv")

NOTEBOOK: ../notebooks/c04_team_seasons.ipynb
CELL: 2
  2:     "S01": (s01, "season", "team"),
  3:     "S02": (s02, "season", "Team"),
  4:     "S03": (s03, "season", "team"),
  5:     "S08": (s08, "Season", "Team_Name")
  6: }
  7: 
  8: # Identify the exact column names in S02 and S03.

NOTEBOOK: ../notebooks/c04_team_seasons.ipynb
CELL: 6
 30:         "team_clean",
 31:         "full_team_name"
 32:     ),
 33:     "S08": (
 34:         s08,
 35:         "Season",
 36:         "Team_Name",
 37:         "short_team_name"

NOTEBOOK: ../notebooks/c04_team_seasons.ipynb
CELL: 7
 10:         validate="one_to_one"
 11:     )
 12: 
 13:     # Standardize season column for S08.
 14:     